# 10 Docker Commands for 90% of Conteraization Tasks

Run all cells inside a disposable Linux VM with a real Docker engine. No host folders, Docker Hub account, paid API or public registry is required. Install Python/ipykernel and (for the command notebook) pexpect and apache2-utils. Resources use unique labels and names; deletion checks identity before acting. Registry credentials are generated only in memory and temporary guest files. Original archive code remains verbatim in a nonexecuting appendix.

In [ ]:
import os, sys, json, uuid, tempfile, subprocess, shutil, platform, hashlib, time
from pathlib import Path
if platform.system() != 'Linux' or not shutil.which('docker'):
    raise RuntimeError('Use a disposable Linux VM with a real running Docker engine.')
endpoint = subprocess.check_output(['docker', 'context', 'inspect', '--format', '{{.Endpoints.docker.Host}}'], text=True).strip()
assert endpoint.startswith('unix://'), 'Only a local engine socket is permitted'
temporary = tempfile.TemporaryDirectory(prefix='tdb-docker-')
work = Path(temporary.name)
config_dir = work / 'docker-config'; config_dir.mkdir()
docker_env = dict(os.environ, DOCKER_HOST=endpoint, DOCKER_CONFIG=str(config_dir))
run_id = uuid.uuid4().hex[:12]; prefix = 'tdb20-' + run_id
owner_label = 'tdb.coding20.run=' + run_id
containers = {}; image_refs = {}; volumes = set(); build_logs = {}
def inspect_container(name):
    p = subprocess.run(['docker', 'inspect', name], env=docker_env, text=True, capture_output=True)
    return json.loads(p.stdout)[0] if p.returncode == 0 else None
def cli(args, *, quiet=False, check=True, input=None, timeout=1200):
    if args[0] in ['stop', 'rm']:
        for name in [x for x in args[1:] if not x.startswith('-')]:
            value = inspect_container(name)
            assert value and value['Config']['Labels'].get('tdb.coding20.run') == run_id
            assert containers.get(name) == value['Id'], 'Container identity changed'
    if args[0] == 'rmi':
        for name in [x for x in args[1:] if not x.startswith('-')]:
            assert name in image_refs
            actual = subprocess.check_output(['docker','image','inspect','--format','{{.Id}}',name],env=docker_env,text=True).strip()
            assert image_refs[name] == actual, 'Image reference changed'
    p = subprocess.run(['docker', *args], cwd=work, env=docker_env, text=True, input=input, capture_output=True, timeout=timeout)
    if check and p.returncode: raise RuntimeError(p.stderr)
    if not quiet:
        print(p.stdout, end='')
        if p.stderr: print('Docker stderr:', p.stderr.strip())
    return p
def image_info(ref): return json.loads(cli(['image','inspect',ref],quiet=True).stdout)[0]
def remember_image(ref):
    image_refs[ref] = image_info(ref)['Id']
    return ref
def create_container(suffix, *, running=True, image='ubuntu:24.04'):
    name = prefix + '-' + suffix
    assert inspect_container(name) is None
    args = ['run','-d'] if running else ['create']
    cli([*args,'--name',name,'--label',owner_label,image,'sleep','1200'],quiet=True)
    containers[name] = inspect_container(name)['Id']
    return name
def tag(source, target):
    assert target.startswith(prefix + '/') or target.startswith('localhost:')
    assert cli(['image','inspect',target],quiet=True,check=False).returncode != 0
    cli(['tag',source,target],quiet=True); remember_image(target)
    return target
def cleanup():
    for name in list(containers):
        if inspect_container(name): cli(['rm','-f','-v',name],quiet=True)
    for name in list(volumes):
        info = json.loads(cli(['volume','inspect',name],quiet=True).stdout)[0]
        assert info['Labels'].get('tdb.coding20.run') == run_id
        cli(['volume','rm',name],quiet=True)
    for ref in list(image_refs):
        if cli(['image','inspect',ref],quiet=True,check=False).returncode == 0:
            cli(['rmi',ref],quiet=True)
    temporary.cleanup()
engine = json.loads(cli(['info','--format','{{json .}}'],quiet=True).stdout)
assert engine['OSType'] == 'linux'
print('Python:', sys.version.split()[0], 'Actual Docker engine:', engine['ServerVersion'])
print('Linux kernel:', platform.release(), 'Architecture:', platform.machine())
print('Only uniquely labelled fixture containers, image references and volumes may be removed.')

In [ ]:
cli(['pull','ubuntu:24.04'],quiet=True)
one=create_container('one')
print('Started fixture:',one)

In [ ]:
print(cli(['run','--rm','--label',owner_label,'ubuntu:24.04','ls'],quiet=True).stdout,end='')

In [ ]:
cli(['ps','--filter','label='+owner_label])

In [ ]:
cli(['ps','-a','--filter','label='+owner_label])

In [ ]:
cli(['ps','--filter','label='+owner_label,'--format','ID: {{.ID}}, Image: {{.Image}}, Status: {{.Status}}'])

In [ ]:
cli(['stop',one])
assert not inspect_container(one)['State']['Running']

In [ ]:
my_container=create_container('my-container')
cli(['stop',my_container])
assert not inspect_container(my_container)['State']['Running']

In [ ]:
three=[create_container('container'+str(i)) for i in range(1,4)]
cli(['stop',*three])
assert all(not inspect_container(name)['State']['Running']for name in three)

In [ ]:
cli(['rm',one])
assert inspect_container(one) is None

In [ ]:
cli(['rm',my_container])
assert inspect_container(my_container) is None

In [ ]:
cli(['rm',*three])
assert all(inspect_container(name) is None for name in three)

In [ ]:
two=[create_container('forced'+str(i)) for i in range(1,3)]
cli(['rm','-f',*two])
assert all(inspect_container(name) is None for name in two)

In [ ]:
repo_tag=tag('ubuntu:24.04',prefix+'/my-repo:my-tag')
image_tag=tag('ubuntu:24.04',prefix+'/my-image:latest')
cli(['images','--filter','reference='+prefix+'/*'])

In [ ]:
cli(['images','--filter','reference='+prefix+'/*'])

In [ ]:
cli(['images',prefix+'/my-repo'])

In [ ]:
cli(['images',repo_tag])

In [ ]:
cli(['images','-a','--filter','reference='+prefix+'/*','--format','table {{.ID}}\t{{.Repository}}\t{{.Tag}}'])

In [ ]:
removable=tag('ubuntu:24.04',prefix+'/removable:latest')
cli(['rmi',removable])
assert cli(['image','inspect',removable],quiet=True,check=False).returncode != 0

In [ ]:
cli(['rmi',image_tag])
assert cli(['image','inspect',image_tag],quiet=True,check=False).returncode != 0

In [ ]:
three_images=[tag('ubuntu:24.04',prefix+'/image'+str(i)+':test')for i in range(1,4)]
cli(['rmi',*three_images])
assert all(cli(['image','inspect',ref],quiet=True,check=False).returncode != 0 for ref in three_images)

In [ ]:
forced_tag=tag('ubuntu:24.04',prefix+'/in-use:latest')
in_use=create_container('in-use',image=forced_tag)
cli(['rmi','-f',forced_tag])
assert inspect_container(in_use)['State']['Running']
print('Only the test tag was removed; its running container still retains image layers.')

In [ ]:
context=work/'build-context';context.mkdir()
(context/'marker.txt').write_text('Docker build fixture\n')
(context/'Dockerfile').write_text('FROM ubuntu:24.04\nCOPY marker.txt /opt/fixture.txt\nCMD ["cat","/opt/fixture.txt"]\n')
built_ref=prefix+'/built:latest'
cli(['build','--label',owner_label,'-t',built_ref,str(context)],quiet=True)
remember_image(built_ref)
print('Built image:',built_ref)

In [ ]:
cli(['build','--label',owner_label,'-t',built_ref,str(context)],quiet=True)
remember_image(built_ref)
print('Actual rebuild succeeded:',built_ref)

In [ ]:
alternate=prefix+'/alternate:latest'
cli(['build','--label',owner_label,'-t',alternate,'-f',str(context/'Dockerfile'),str(context)],quiet=True)
remember_image(alternate)
assert cli(['run','--rm','--label',owner_label,alternate],quiet=True).stdout == 'Docker build fixture\n'
print('Alternate Dockerfile path built and ran correctly.')

In [ ]:
exec_container=create_container('exec')
cli(['exec',exec_container,'cat','/etc/os-release'])

In [ ]:
cli(['exec',exec_container,'ls','/'])

In [ ]:
import pexpect, re
child=pexpect.spawn('docker',['exec','-it',exec_container,'bash'],env=docker_env,encoding='utf-8',timeout=20)
child.sendline('printf "PTY_RESULT=%s\\n" "$((3+4))"; exit')
child.expect(pexpect.EOF)
transcript=re.sub(r'\x1b\[[0-9;?]*[A-Za-z]','',child.before).replace('\r','')
child.close()
assert child.exitstatus == 0 and re.search(r'^PTY_RESULT=7$',transcript,re.M)
print('Real interactive docker exec -it shell computed PTY_RESULT=7.')

In [ ]:
cli(['pull','ubuntu:24.04'],quiet=True)
print('Actual pull digest:',image_info('ubuntu:24.04')['RepoDigests'][0])

In [ ]:
cli(['pull','ubuntu:24.04'],quiet=True)
print('Supported, explicit Ubuntu tag pulled successfully.')

In [ ]:
cli(['pull','ubuntu:22.04'],quiet=True)
assert image_info('ubuntu:22.04')['RepoDigests']
print('A second explicit supported tag was pulled:',image_info('ubuntu:22.04')['RepoDigests'][0])

In [ ]:
import secrets, socket, urllib.request, base64
cli(['pull','registry:3'],quiet=True)
with socket.socket() as probe:
    probe.bind(('127.0.0.1',0));registry_port=probe.getsockname()[1]
auth_dir=work/'registry-auth';auth_dir.mkdir()
test_password=secrets.token_urlsafe(24)
hashed=subprocess.check_output(['htpasswd','-Bbn','tutorial',test_password],text=True)
(auth_dir/'htpasswd').write_text(hashed)
registry_name=prefix+'-registry'
cli(['run','-d','--name',registry_name,'--label',owner_label,'-p',f'127.0.0.1:{registry_port}:5000','--mount',f'type=bind,source={auth_dir},target=/auth,readonly','-e','REGISTRY_AUTH=htpasswd','-e','REGISTRY_AUTH_HTPASSWD_REALM=Tutorial','-e','REGISTRY_AUTH_HTPASSWD_PATH=/auth/htpasswd','registry:3'],quiet=True)
containers[registry_name]=inspect_container(registry_name)['Id']
registry_address=f'localhost:{registry_port}'
for attempt in range(60):
    p=cli(['login',registry_address,'--username','tutorial','--password-stdin'],quiet=True,check=False,input=test_password+'\n')
    if p.returncode==0:break
    time.sleep(0.5)
else:raise RuntimeError(p.stderr)
wrong=cli(['login',registry_address,'--username','tutorial','--password-stdin'],quiet=True,check=False,input='incorrect-test-password\n')
assert wrong.returncode != 0
cli(['login',registry_address,'--username','tutorial','--password-stdin'],quiet=True,input=test_password+'\n')
local_ref=tag(built_ref,registry_address+'/'+prefix+'/my-image:1.0')
cli(['push',local_ref],quiet=True)
first_digest=image_info(local_ref)['RepoDigests']
assert first_digest
cli(['rmi',local_ref],quiet=True)
cli(['pull',local_ref],quiet=True);remember_image(local_ref)
assert cli(['run','--rm','--label',owner_label,local_ref],quiet=True).stdout=='Docker build fixture\n'
print('Actual authenticated push/pull round-trip to a loopback-only registry passed; wrong credentials were rejected.')
print('Registry manifest digest:',first_digest[0].rsplit('@',1)[-1])

In [ ]:
second_ref=tag(built_ref,registry_address+'/'+prefix+'/my-image:latest')
cli(['push',second_ref],quiet=True)
cli(['pull',second_ref],quiet=True)
assert image_info(second_ref)['Id'] == image_info(built_ref)['Id']
print('The latest tag was also pushed and verified against the actual built image ID. No public registry received test data.')

In [ ]:
assert image_info(local_ref)['Id'] == image_info(built_ref)['Id']
assert all(inspect_container(name) is None or inspect_container(name)['Config']['Labels'].get('tdb.coding20.run')==run_id for name in containers)
print('PASS: all ten command families, real lifecycle/forced deletion, builds, interactive exec, pulls, authenticated loopback registry push/pull and identity checks.')
cleanup()
print('Removed only this run’s labelled containers, aliases, test registry and temporary credentials.')

## Original source-code appendix

Every original block is retained verbatim, including generic syntax and incomplete Dockerfiles. Use the tested examples above.

```bash
docker run [OPTIONS] IMAGE [COMMAND] [ARG...]
```

```bash
docker run ubuntu ls
```

```bash
docker ps [OPTIONS]
```

```bash
docker ps -a
```

```bash
docker ps --format "ID: {{.ID}}, Image: {{.Image}}, Status: {{.Status}}"
```

```bash
docker stop [OPTIONS] CONTAINER [CONTAINER...]
```

```bash
docker stop my-container
```

```bash
docker stop container1 container2 container3
```

```bash
docker rm [OPTIONS] CONTAINER [CONTAINER...]
```

```bash
docker rm my-container
```

```bash
docker rm container1 container2 container3
```

```bash
docker rm -f container1 container2
```

```bash
docker images [OPTIONS] [REPOSITORY[:TAG]]
```

```bash
docker images
```

```bash
docker images my-repo
```

```bash
docker images my-repo:my-tag
```

```bash
docker images -a --format "table {{.ID}}\t{{.Repository}}\t{{.Tag}}"
```

```bash
docker rmi [OPTIONS] IMAGE [IMAGE...]
```

```bash
docker rmi my-image:latest
```

```bash
docker rmi image1 image2 image3
```

```bash
docker rmi -f my-image:latest
```

```bash
docker build [OPTIONS] PATH | URL | -
```

```bash
docker build -t my-image:latest .
```

```bash
docker build -t my-image:latest -f /path/to/Dockerfile .
```

```bash
docker exec [OPTIONS] CONTAINER COMMAND [ARG...]
```

```bash
docker exec my-container ls
```

```bash
docker exec -it my-container bash
```

```bash
docker pull [OPTIONS] IMAGE[:TAG]
```

```bash
docker pull ubuntu
```

```bash
docker pull ubuntu:20.04
```

```bash
docker push [OPTIONS] NAME[:TAG]
```

```bash
docker push username/my-image:latest
```